# Detección de objetos con YOLO26
### El detector moderno: entrenar y usar en pocas líneas

**Universidad de Santiago de Chile (USACH)**  
Diplomado de Inteligencia artificial aplicada a la industria tecnológica  
Autor: Diego Fischer

Flujo del cuaderno:

1. Instalar y probar un modelo preentrenado (inferencia inmediata).
2. Entender el **formato de dataset de YOLO** (imágenes, labels `.txt` y `data.yaml`).
3. **Entrenar** el modelo con tu propio dataset.
4. Revisar las métricas, **validar** y **usar** tu modelo entrenado.
5. **Exportar** el modelo para producción.
---

## ¿Qué hace especial a YOLO26?

YOLO (*You Only Look Once*) es la familia de detectores *one-stage* más famosa: mira la imagen completa una sola vez y predice todos los objetos de golpe. La versión 26 introduce dos simplificaciones importantes respecto de sus antecesores.

### 1. Inferencia *end-to-end*, sin NMS

¿Recuerdas el **NMS** (*Non-Maximum Suppression*) de los cuadernos anteriores? Era el paso de post-procesamiento que eliminaba cajas duplicadas apuntando al mismo objeto: el modelo proponía varias cajas casi iguales y NMS se quedaba con la mejor.

YOLO26 **elimina ese paso**. Su cabeza "uno-a-uno" (*one-to-one*) aprende a predecir **una sola caja por objeto** desde el principio, así que no hacen falta reglas externas para limpiar duplicados. Menos post-procesamiento significa menor latencia y despliegue más simple — clave para dispositivos de borde (drones, celulares, sistemas embebidos). Por defecto entrega hasta 300 detecciones por imagen, ya "limpias".

> Si necesitas el comportamiento clásico (cabeza *one-to-many* + NMS, que a veces exprime algo más de precisión), YOLO26 lo permite con `end2end=False`. Tiene una arquitectura de **doble cabeza**.

### 2. Regresión sin DFL y entrenamiento más estable

YOLO26 quita la *Distribution Focal Loss* (DFL) de la cabeza de regresión, simplificando la exportación a formatos como ONNX o TensorRT. Y usa un optimizador nuevo, **MuSGD** (una mezcla de SGD con ideas del optimizador Muon, tomado del entrenamiento de modelos de lenguaje grandes), que da convergencia más estable. También mejora la detección de **objetos pequeños** mediante técnicas llamadas ProgLoss y STAL.

### La filosofía Ultralytics: una sola clase, un solo comando

Todo en YOLO26 pasa por una única clase, `YOLO`, y unos pocos métodos:

| Acción | Método | Equivalente "a mano" en torchvision |
|---|---|---|
| Cargar | `YOLO('yolo26n.pt')` | cargar pesos + reemplazar cabeza |
| Entrenar | `model.train(...)` | escribir Dataset + bucle de entrenamiento |
| Predecir | `model(imagen)` | preprocesar + forward + postproceso |
| Validar | `model.val(...)` | implementar cálculo de mAP |
| Exportar | `model.export(...)` | conversión manual a ONNX |

Cada celda de este cuaderno mostrará lo poco que hace falta escribir. Esa es justamente la gracia — y también su riesgo pedagógico: es tan fácil que uno puede usarlo sin entender qué pasa por dentro. Por eso lo dejamos para el final de la serie, cuando ya viste las tripas.

---

## Fase 1 — Instalación y entorno

A diferencia de `torchvision` (que viene con Colab), el paquete `ultralytics` hay que instalarlo. Es una sola línea. Al instalarlo se traen también todas las dependencias necesarias.

> **Importante:** activa la **GPU** antes de entrenar (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*). Entrenar YOLO en CPU es muy lento.

In [ ]:
!pip install -q -U ultralytics

In [ ]:
import ultralytics
from ultralytics import YOLO
import torch

ultralytics.checks()  # imprime versiones, GPU disponible, etc.
print('\n¿GPU disponible?:', torch.cuda.is_available())

## Fase 2 — Probar un modelo preentrenado (inferencia inmediata)

Antes de entrenar nada, veamos la magia del "todo en una línea". Cargamos `yolo26n.pt` (la variante *nano*, la más liviana) — que se **descarga sola** la primera vez — y la aplicamos sobre una imagen. Viene preentrenada en COCO, así que ya reconoce 80 clases cotidianas.

Las variantes van de menor a mayor tamaño y precisión: `n` (nano), `s` (small), `m` (medium), `l` (large), `x` (extra-large). Más grande = más preciso pero más lento. Para aprender y para *edge*, `n` es ideal.

In [ ]:
# Cargar modelo preentrenado (se descarga automáticamente la 1ra vez)
modelo = YOLO('yolo26n.pt')

# Inferencia sobre una imagen de ejemplo (acepta ruta local, URL o carpeta)
resultados = modelo('https://ultralytics.com/images/bus.jpg')

print(f'Objetos detectados: {len(resultados[0].boxes)}')

In [ ]:
import matplotlib.pyplot as plt

# .plot() dibuja las cajas y devuelve la imagen anotada (en formato BGR)
imagen_anotada = resultados[0].plot()

plt.figure(figsize=(8, 10))
plt.imshow(imagen_anotada[..., ::-1])  # BGR -> RGB para matplotlib
plt.axis('off')
plt.title('YOLO26n — inferencia sobre imagen de ejemplo')
plt.show()

### Leer las detecciones como datos

El objeto `resultados[0]` no es solo una imagen bonita: contiene los datos crudos de cada detección, que puedes usar programáticamente (contar objetos, filtrar por clase, disparar una alerta, etc.).

In [ ]:
r = resultados[0]

print(f"{'Clase':<15}{'Confianza':<12}{'Caja (x1,y1,x2,y2)'}")
print('-' * 55)
for caja in r.boxes:
    id_clase = int(caja.cls[0])
    nombre = r.names[id_clase]          # r.names traduce id -> nombre
    conf = float(caja.conf[0])
    x1, y1, x2, y2 = [round(v) for v in caja.xyxy[0].tolist()]
    print(f'{nombre:<15}{conf:<12.3f}({x1}, {y1}, {x2}, {y2})')

> **Fíjate:** no llamamos a ningún NMS ni filtramos duplicados a mano. Las cajas llegan ya "limpias" gracias a la inferencia *end-to-end* de YOLO26. Compáralo con el trabajo de post-procesamiento que discutíamos en los cuadernos de torchvision.

## Fase 3 — El formato de dataset de YOLO

Para entrenar con **tus** objetos necesitas tus imágenes anotadas en el formato que YOLO espera. Es distinto del CSV que usamos en el cuaderno de torchvision, así que vale la pena entenderlo bien (aunque el dataset ya te llegue armado).

### La estructura de carpetas

```
mi_dataset/
├── images/
│   ├── train/          <- imágenes de entrenamiento
│   │   ├── img001.jpg
│   │   └── ...
│   └── val/            <- imágenes de validación
│       └── ...
├── labels/
│   ├── train/          <- un .txt por CADA imagen de train
│   │   ├── img001.txt
│   │   └── ...
│   └── val/
│       └── ...
└── data.yaml           <- el archivo de configuración
```

YOLO empareja cada imagen con su label buscando el `.txt` del **mismo nombre** en la carpeta `labels` correspondiente (`images/train/img001.jpg` → `labels/train/img001.txt`).

### El formato de cada archivo de label (`.txt`)

Cada imagen tiene un `.txt` con **una línea por objeto**, y cada línea tiene cinco números:

```
<id_clase>  <x_centro>  <y_centro>  <ancho>  <alto>
```

Por ejemplo, `img001.txt` con dos objetos:

```
0  0.512  0.430  0.233  0.180
1  0.700  0.660  0.120  0.240
```

Dos diferencias **cruciales** respecto del formato de torchvision del cuaderno anterior:

- **Coordenadas normalizadas (0 a 1), no en píxeles.** Cada valor se divide por el tamaño de la imagen: `x_centro = px_centro / ancho_imagen`. La ventaja: el label no depende de si luego redimensionas la imagen.
- **Centro + tamaño, no dos esquinas.** YOLO describe la caja por su **centro** `(x_centro, y_centro)` y su `(ancho, alto)`, en vez de las esquinas `(x1,y1,x2,y2)`.

Y una diferencia de convención que **es fuente clásica de errores**: en YOLO el `id_clase` es **0-indexado y el 0 es una clase real** (la primera). No hay clase de "fondo" reservada como en torchvision. Si tienes 2 clases, sus ids son 0 y 1.

> **Conversión de esquinas a formato YOLO** (por si alguna vez lo necesitas a mano), para una imagen de `W×H` píxeles:
> ```
> x_centro = ((x1 + x2) / 2) / W        ancho = (x2 - x1) / W
> y_centro = ((y1 + y2) / 2) / H        alto  = (y2 - y1) / H
> ```

### El archivo `data.yaml`

Es el "índice" que le dice a YOLO dónde están los datos y qué significa cada id de clase:

```yaml
# data.yaml
path: /content/mi_dataset   # carpeta raíz del dataset
train: images/train         # imágenes de train (relativo a 'path')
val: images/val             # imágenes de val

# nombres de las clases: el id 0 es la primera, el 1 la segunda, etc.
names:
  0: gato
  1: perro
```

El orden de `names` **define** el mapeo id→nombre, y debe coincidir con los `id_clase` que usaste en los `.txt`. Como tú les entregas el dataset ya con labels y yaml, tus estudiantes solo tendrán que apuntar a ese `data.yaml` — que es justo lo que haremos en la siguiente fase.

### Herramientas para anotar

Si algún día necesitan crear sus propios labels en este formato: *makesense.ai* (en el navegador), *LabelImg*, *CVAT* o *Roboflow* exportan directamente al formato YOLO.

## Fase 4 — Cargar tu dataset y entrenar

Aquí eliges entre dos caminos con la variable `USAR_DEMO`:

- **`True` (demo):** usamos `coco8.yaml`, un dataset de juguete de 8 imágenes que Ultralytics **descarga solo**. Sirve para que todo el pipeline corra en segundos y veas el flujo completo. (Con tan pocas imágenes el modelo se "sobreajusta" al instante; es solo para demostrar el mecanismo, no para obtener un buen detector.)
- **`False` (tu dataset):** subes el ZIP de tu dataset (con la estructura de la Fase 3) y apuntas a tu `data.yaml`.

In [ ]:
import os, zipfile, io

USAR_DEMO = True   # <-- cambia a False para usar TU dataset

if USAR_DEMO:
    DATASET_YAML = 'coco8.yaml'   # se descarga automáticamente
    print('Usando dataset de demostración: coco8')
else:
    from google.colab import files
    print('Sube el .zip de tu dataset (images/, labels/ y data.yaml dentro)...')
    subido = files.upload()
    nombre_zip = list(subido.keys())[0]
    with zipfile.ZipFile(io.BytesIO(subido[nombre_zip])) as z:
        z.extractall('/content/mi_dataset')
    # Buscamos el data.yaml dentro de lo que se descomprimió
    DATASET_YAML = None
    for raiz, _, archivos in os.walk('/content/mi_dataset'):
        for a in archivos:
            if a.endswith('.yaml'):
                DATASET_YAML = os.path.join(raiz, a)
    assert DATASET_YAML, 'No encontré un .yaml dentro del ZIP'
    print('Tu dataset:', DATASET_YAML)
    print('\nRevisa que las rutas train/val dentro del yaml coincidan con la')
    print('estructura descomprimida. Si usan "path:", ajústalo a /content/mi_dataset.')

### Entrenar

El entrenamiento es **una sola llamada**: `model.train(...)`. Igual que en los cuadernos anteriores, esto es *transfer learning* — partimos de los pesos COCO, así que el modelo ya sabe ver y solo lo afinamos a tus clases.

Parámetros que conviene conocer:

- **`data`**: la ruta a tu `data.yaml`.
- **`epochs`**: cuántas veces recorrer el dataset. Pocas para la demo; para datos reales, típicamente 50–300.
- **`imgsz`**: tamaño al que se redimensionan las imágenes (640 es el estándar).
- **`batch`**: cuántas imágenes por paso (baja este número si te quedas sin memoria de GPU).
- **`patience`**: si la métrica no mejora en tantas épocas, se detiene solo (*early stopping*).

Todo lo que produce el entrenamiento (pesos, gráficos, métricas) se guarda en `runs/detect/train/`.

In [ ]:
# Cargamos el modelo base y entrenamos
modelo = YOLO('yolo26n.pt')

NUM_EPOCAS = 15   # sube esto bastante para datos reales

resultados_train = modelo.train(
    data=DATASET_YAML,
    epochs=NUM_EPOCAS,
    imgsz=640,
    batch=16,
    patience=20,
    plots=True,          # genera gráficos de métricas
    verbose=True,
)

print('\nEntrenamiento terminado.')
print('Carpeta de resultados:', modelo.trainer.save_dir)

## Fase 5 — Revisar los resultados del entrenamiento

YOLO genera automáticamente un montón de gráficos útiles en la carpeta de resultados. Los dos más importantes:

- **`results.png`**: la evolución de las pérdidas y de las métricas (mAP, precisión, *recall*) a lo largo de las épocas. Es tu "tablero de control": las pérdidas deben bajar y las métricas subir.
- **`confusion_matrix.png`**: qué clases confunde el modelo con cuáles.

Los mostramos aquí mismo.

In [ ]:
from pathlib import Path
from PIL import Image as PILImage

carpeta = Path(modelo.trainer.save_dir)

for archivo in ['results.png', 'confusion_matrix.png']:
    ruta = carpeta / archivo
    if ruta.exists():
        img = PILImage.open(ruta)
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(archivo)
        plt.show()
    else:
        print('No se encontró', archivo)

### ¿Qué métricas mirar?

En los cuadernos anteriores advertíamos que "contar detecciones no mide calidad" y que la métrica seria es el **mAP**. YOLO lo calcula por ti:

- **mAP50**: *mean Average Precision* con umbral de solapamiento IoU de 0.5. Más indulgente.
- **mAP50-95**: promedia el mAP sobre varios umbrales de IoU (de 0.5 a 0.95). Es la métrica **estándar y más exigente** de la literatura (la de COCO).

Un buen modelo tiene ambas altas; mAP50-95 alto significa que además de encontrar el objeto, lo localiza con cajas muy ajustadas.

## Fase 6 — Validar el modelo

`model.val()` corre el modelo sobre el conjunto de validación definido en tu yaml y reporta las métricas numéricas. Sirve para comparar modelos o configuraciones de forma objetiva.

In [ ]:
metricas = modelo.val()

print(f'mAP50-95 : {metricas.box.map:.4f}')
print(f'mAP50    : {metricas.box.map50:.4f}')
print(f'mAP75    : {metricas.box.map75:.4f}')

## Fase 7 — Usar tu modelo entrenado

El entrenamiento guarda dos juegos de pesos en `runs/detect/train/weights/`:

- **`best.pt`**: los pesos de la época con mejor métrica de validación. **Este es el que usarás** casi siempre.
- **`last.pt`**: los de la última época (útil para reanudar entrenamiento).

Cargamos `best.pt` y lo aplicamos sobre imágenes nuevas.

In [ ]:
ruta_mejor = carpeta / 'weights' / 'best.pt'
mi_modelo = YOLO(ruta_mejor)
print('Modelo entrenado cargado desde:', ruta_mejor)

# Predecir sobre una imagen. Con TU dataset, usa una imagen NUEVA (no de train).
# Para la demo reutilizamos la imagen de ejemplo.
pred = mi_modelo('https://ultralytics.com/images/bus.jpg', conf=0.25)

plt.figure(figsize=(8, 10))
plt.imshow(pred[0].plot()[..., ::-1])
plt.axis('off')
plt.title('Detecciones de tu modelo entrenado')
plt.show()

> **El parámetro `conf`** es el umbral de confianza (lo mismo que la "perilla" de los cuadernos anteriores): sube `conf` para ver solo detecciones muy seguras, bájalo para ver más candidatas. También puedes pasarle una **carpeta** de imágenes o incluso un **video** (`.mp4`) y las procesa todas.

In [ ]:
# Para subir tus propias imágenes de prueba y correr inferencia:
#
# from google.colab import files
# subidas = files.upload()
# for nombre in subidas:
#     pred = mi_modelo(nombre, conf=0.25)
#     plt.figure(figsize=(8, 8))
#     plt.imshow(pred[0].plot()[..., ::-1])
#     plt.axis('off'); plt.show()

print('Descomenta el bloque de arriba para probar con tus imágenes.')

## Fase 8 — Exportar para producción

YOLO26 fue diseñado pensando en el **despliegue en dispositivos de borde**, y su diseño *NMS-free / DFL-free* hace que la exportación sea especialmente limpia. Con una línea conviertes tu modelo a formatos optimizados para distintos destinos:

- **ONNX**: formato universal, corre en casi cualquier runtime.
- **TensorRT**: máxima velocidad en GPUs NVIDIA (Jetson).
- **CoreML**: para iOS.
- **TFLite / OpenVINO**: para móviles y CPUs de borde.

Exportamos a ONNX como ejemplo.

In [ ]:
ruta_onnx = mi_modelo.export(format='onnx')
print('Modelo exportado a:', ruta_onnx)

# Para descargar los pesos entrenados (.pt) a tu computador:
#   from google.colab import files
#   files.download(str(carpeta / 'weights' / 'best.pt'))

---
## Licencia

[![Licencia Creative Commons](https://i.creativecommons.org/l/by/4.0/88x31.png)](https://creativecommons.org/licenses/by/4.0/deed.es)

Este cuaderno está licenciado bajo una [Licencia Creative Commons Atribución 4.0 Internacional (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/deed.es). Puede compartirse y adaptarse, incluso con fines comerciales, siempre que se reconozca la autoría de forma adecuada.

**Atribución:** Diego Fischer — Universidad de Santiago de Chile (USACH).

> Nota: el material de este cuaderno (texto y código de ejemplo) es de Diego Fischer bajo CC BY. El *software* YOLO26 y sus pesos, en cambio, son de Ultralytics y se rigen por su propia licencia **AGPL-3.0** (o su licencia empresarial), independiente de la licencia de este cuaderno.